**Act 2 to Act 3.** Act 2 said the GBSA-locked baseline holds up. No MM-GBSA parameter tweak or MD (molecular dynamics) runtime perturbation moved panel BEDROC (early-enrichment score, α=20) more than the bootstrap CI half-width. The L27 Taguchi screen put main-effect standard deviations below the panel-BEDROC noise floor.

Now Act 3 asks the next question: **what does an MD trajectory look like frame by frame at the top vs bottom of the panel, and which of those observables aggregate into something a ranker can use?**

NBs 14–21 build the answer bottom up. Per-complex worked example (14), per-target aggregation (15), feature distributions split by is-active (16), kinematic phase space (17), time-series overlays (18), active-site fingerprint (19), correlation block (20), actives-vs-decoys separation (21). Skip ahead to 22 if you only want the ranking verdict — but the feature list Act 4 uses is defined here.


> **Reader guide.** *Experiment A3 (see [STUDY_DESIGN §A3](../../STUDY_DESIGN.md)):* per-complex
> MD-stability feature extraction — the raw form for one complex, so every summary column that
> the later panel plots aggregate is inspectable end-to-end.
>
> **Method:** 8-panel timeseries + contacts + RMSF for one selected complex.
>
> **Reproducibility contract:** reads `data/raw/complex_analyses/{TARGET}/{ID}/`; plots are
> direct timeseries, no CSV persistence needed for aggregate.

# 14 — Per-complex deep dive

One representative complex, unrolled. Eight time-series panels (drift, RMSF = root-mean-square fluctuation, H-bonds, contacts, radius of gyration) so you can see every summary column in raw form before we aggregate. This is the Act 3 opener. Everything after it (per-target means, phase spaces, overlays) collapses this kind of plot into one number per complex.

_(Notebook auto-generated by `reproduce/split_monolith.py`. Self-contained: loads its data via `discovery9.io`, exports figures to `figures/14_per_complex_deep_dive_figK.png`.)_


In [ ]:
# --- notebook preamble ---
NB_STEM = "30_per_complex_deep_dive"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 2. Per-complex deep dive — one complex, all outputs

Before touching aggregates, we need to *see* what one trajectory does. Every summary metric collapses one of these time series into a number, and reductions hide behaviour like escape-then-return, secondary-mode flips, or slow SASA breathing.

The 8 panels are the canonical MD-stability signals for one complex over 30 ns. "Flat = stable" means:

| Panel | Flat = stable looks like |
|---|---|
| `rmsd_bb_A` | equilibrates in the first ~2 ns, then stays within ±0.5 Å around 1–2 Å |
| `rmsd_as_bb_A` | usually **lower** than global BB (active site pinned by the ligand) |
| `lig_drift_A` | real binder: under 2–3 Å for the whole 30 ns |
| `lig_com_disp_A` | mirror of drift; escape events show as monotonic climbs |
| `lig_buried_sasa_A2` | fluctuates around a plateau; a drop toward free-ligand SASA = exposed |
| `vdw_contacts` | > ~30 for a well-buried ligand; sudden drops = pose flip |
| `protein_rg_A` | very flat (~ ±0.1 Å); breathing but no collapse |
| `lig_internal_rmsd_A` | flat = rigid pose, sawtooth = torsional flipping |


In [ ]:

TARGET = '4QB3'  # ← edit
sub = df[df.target == TARGET]
if 'is_active' in df and (sub.is_active == True).any():
    COMPLEX_ID = sub[sub.is_active == True].iloc[0].complex_id
    print(f'Picked first known-active for {TARGET}: {COMPLEX_ID}')
else:
    COMPLEX_ID = sub.iloc[0].complex_id
    print(f'No labeled active in scope — picked first {TARGET} complex: {COMPLEX_ID}')
cdir = os.path.join(str(RAW / 'complex_analyses'), TARGET, COMPLEX_ID)

summary = json.load(open(os.path.join(cdir, 'summary.json')))
act = sub[sub.complex_id == COMPLEX_ID].is_active.iloc[0] if 'is_active' in df else 'unknown'
print(f'\n  is_active               : {act}')
print(f"  # active-site residues  : {summary['n_active_site_residues']}")
print(f"  ligand heavy atoms      : {summary['n_lig_heavy']}")
print(f"  ligand net charge       : {summary['ligand_partial_charge_sum']:+.2f} e")
print(f"  protein formal charge   : {summary['protein_formal_charge']:+d} e  (over {summary['protein_n_titratable']} titratable residues)")
print(f"  active-site formal q    : {summary['active_site_formal_charge']:+d} e")
print(f"  mean pose drift         : {summary['lig_drift_mean_A']:.2f} Å  (escape fraction: {summary['lig_escape_frac']:.2f})")
print(f"  mean # H-bonds          : {summary['n_hb_mean']:.2f}  (persistence: {summary['hb_persistence_frac']*100:.0f}% of frames)")

In [ ]:

ts = pd.read_parquet(os.path.join(cdir, 'timeseries.parquet'))
t = ts.time_ps / 1000

fig, axes = plt.subplots(4, 2, figsize=(13, 12), sharex=True)
panels = [
    ('rmsd_bb_A',           'Protein backbone RMSD',     'Å',  NAVY),
    ('rmsd_as_bb_A',        'Active-site backbone RMSD', 'Å',  NAVY),
    ('lig_drift_A',         'Ligand pose drift',         'Å',  GOLD),
    ('lig_com_disp_A',      'Ligand COM displacement',   'Å',  GOLD),
    ('lig_buried_sasa_A2',  'Ligand buried SASA',        'Å²', NAVY),
    ('vdw_contacts',        'vdW contacts (<4 Å)',       '#',  NAVY),
    ('protein_rg_A',        'Protein Rg',                'Å',  GREYD),
    ('lig_internal_rmsd_A', 'Ligand internal RMSD',      'Å',  GOLD),
]
for ax, (col, title, unit, color) in zip(axes.ravel(), panels):
    ax.plot(t, ts[col], lw=1.0, color=color)
    ax.fill_between(t, ts[col], alpha=0.18, color=color)
    ax.set_title(f'{title}  [{unit}]', fontsize=10)
    ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
for ax in axes[-1]:
    ax.set_xlabel('time [ns]')
fig.suptitle(f'{TARGET} · {COMPLEX_ID[:12]}…  — 30 ns trajectory', y=1.00, fontsize=12, color=NAVY, fontweight='bold')
plt.tight_layout()

In [ ]:

hb_p = os.path.join(cdir, 'hbond_timeseries.parquet')
if not os.path.exists(hb_p):
    print('no HB time series file (analysis may have failed for this complex)')
else:
    hb = pd.read_parquet(hb_p)
    fig, ax = plt.subplots(figsize=(11, 3.4))
    ax.plot(hb.time_ps/1000, hb.n_hb, lw=1.0, color=GOLD)
    ax.fill_between(hb.time_ps/1000, hb.n_hb, alpha=0.35, color=GOLD)
    ax.set_xlabel('time [ns]'); ax.set_ylabel('# H-bonds')
    ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
    ax.set_title(f'{TARGET} · ligand ↔ active-site H-bonds  (mean = {hb.n_hb.mean():.2f}, persistence = {(hb.n_hb>0).mean()*100:.0f}% of frames)')

In [ ]:

cp = pd.read_csv(os.path.join(cdir, 'contacts_persistence.tsv'), sep='\t').sort_values('persistence', ascending=True)
cp_top = cp.tail(min(20, len(cp)))
fig, ax = plt.subplots(figsize=(9, max(4, 0.28*len(cp_top))))
labels = [f'{int(r)} {n}' for r,n in zip(cp_top.resid, cp_top.resname)]
colors = [GOLD if p > 0.8 else (NAVY if p > 0.5 else GREYD) for p in cp_top.persistence]
ax.barh(labels, cp_top.persistence, color=colors, edgecolor=WHITE, linewidth=0.6)
ax.set_xlim(0, 1); ax.axvline(0.5, color=GREY, ls=':', lw=1); ax.axvline(0.8, color=GREY, ls=':', lw=1)
ax.set_xlabel('persistence (fraction of frames with min-dist < 4.5 Å)')
ax.set_title(f'{TARGET} · top active-site contact persistence  (GOLD > 0.8, NAVY > 0.5, grey < 0.5)')
ax.set_axisbelow(True); ax.xaxis.grid(True, color=GREY, alpha=0.5)

In [ ]:

lig_rmsf = pd.read_parquet(os.path.join(cdir, 'ligand_rmsf.parquet'))
fig, ax = plt.subplots(figsize=(11, 3.4))
colors_by_el = {'C': NAVY, 'N': GOLD, 'O': GOLD, 'S': GOLD, 'F': GREYD, 'P': GREYD, 'Cl': GREYD}
colors = [colors_by_el.get(e, GREYD) for e in lig_rmsf.element]
ax.bar(range(len(lig_rmsf)), lig_rmsf.rmsf_A, color=colors, edgecolor=WHITE, linewidth=0.6)
ax.set_xticks(range(len(lig_rmsf)))
ax.set_xticklabels(lig_rmsf.atomname, rotation=90, fontsize=7)
ax.set_ylabel('RMSF [Å]')
ax.set_title(f'{TARGET} · ligand per-atom RMSF  (NAVY = C, GOLD = N/O/S, grey = halogen/P)')
ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)

In [ ]:

ca = pd.read_parquet(os.path.join(cdir, 'protein_ca_rmsf.parquet'))
fig, ax = plt.subplots(figsize=(12, 3.4))
colors = [GOLD if a else NAVY for a in ca.is_active_site]
ax.bar(ca.resid, ca.ca_rmsf_A, color=colors, width=1.0)
ax.set_xlabel('residue'); ax.set_ylabel('Cα RMSF [Å]')
ax.set_title(f'{TARGET} · protein Cα RMSF  (GOLD = active-site residues)')
ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)

**What we read off.** A good binder shows all four ligand panels flat: `lig_drift < 3 Å`, `lig_com_disp` flat, `vdw_contacts > 30`, `lig_buried_sasa` on a plateau. If `lig_drift` climbs and stays up, or `lig_com_disp` walks off monotonically, the ligand left the pocket.

The contact-persistence bar chart is also a pharmacophore signature. GOLD bars are residues the ligand actually engages, not the ones docking placed it near. Compare this residue set to the pocket residues in the target's crystal structure — that's a cheap sanity check on the docking box.


In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
